# Validación y Manejo de Errores del Pipeline ETL

Este notebook demuestra el uso del módulo `etl_validation.py` para:
1. Cargar el dataset crudo
2. Ejecutar validaciones de esquema
3. Evaluar calidad de datos
4. Registrar todo en logs profesionales

**Audiencia:** Data Engineers, MLOps

**Archivo de log:** `etl_pipeline.log`

## Importar módulos

In [1]:
import pandas as pd
import sys
from pathlib import Path

# Agregar src al path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Importar funciones de validación
from etl_validation import (
    setup_etl_logger,
    validate_schema,
    validate_data_quality,
    SCHEMA_RAW,
    SCHEMA_CLEAN,
    handle_etl_errors
)

print("✅ Módulos importados correctamente")

✅ Módulos importados correctamente


## Configurar logger

In [2]:
# Configurar logger profesional
logger = setup_etl_logger(log_file='etl_pipeline.log')
logger.info("="*70)
logger.info("Iniciando proceso de validación del pipeline ETL")
logger.info("="*70)

2026-06-29 19:59:01 - ETL_Pipeline - INFO - ======================================================================
2026-06-29 19:59:01 - ETL_Pipeline - INFO - Iniciando proceso de validación del pipeline ETL
2026-06-29 19:59:01 - ETL_Pipeline - INFO - ======================================================================


## Etapa 1: Cargar dataset crudo

In [4]:
from Routes import RUTAS

try:
    ruta_raw = RUTAS['data_raw'] / 'Dirty_Sales_Marketing_Dataset.xlsx'
    logger.info(f"Cargando dataset crudo desde: {ruta_raw}")
    
    df_raw = pd.read_excel(ruta_raw)
    logger.info(f"✅ Dataset cargado: {df_raw.shape[0]} filas, {df_raw.shape[1]} columnas")
    
except FileNotFoundError as e:
    logger.error(f"❌ Archivo no encontrado: {ruta_raw}")
    raise
except Exception as e:
    logger.error(f"❌ Error al cargar dataset: {str(e)}")
    raise

print(df_raw.head())

2026-06-29 19:59:31 - ETL_Pipeline - INFO - Cargando dataset crudo desde: c:\Users\encar\Desktop\Proyectos_Ciencia_de_Datos\Sales_Marketing_Dataset\data\raw\Dirty_Sales_Marketing_Dataset.xlsx
2026-06-29 19:59:34 - ETL_Pipeline - INFO - ✅ Dataset cargado: 15750 filas, 30 columnas


  customer_id    gender     age  country     city signup_date  \
0    10001.0      MALE     nan     India   Berlin  2022-05-10   
1    10002.0       nan    35.0   Germany   Mumbai  2024-06-16   
2    10003.0    female    27.0   Germany      NaN  2023-08-23   
3    10004.0    FEMALE    36.0     India   Mumbai  2024-01-28   
4    10005.0      MALE    29.0       USA  Hamburg  2023-07-21   

  last_purchase_date acquisition_channel device_type subscription_type  ...  \
0         2024-12-31               Email      Tablet            Annual  ...   
1         2024-05-07             Organic     Desktop           Monthly  ...   
2         2024-04-28               Email      Mobile            Annual  ...   
3         2023-05-20        Facebook Ads      Tablet            Annual  ...   
4         2024-04-07            Referral      Mobile           Monthly  ...   

   support_tickets  refund_requested  delivery_delay_days  payment_method  \
0                0                 0                    3

## Etapa 2: Validar esquema del dataset crudo

In [ ]:
# Convert age column to numeric, coercing errors to NaN
df_raw['age'] = pd.to_numeric(df_raw['age'], errors='coerce')

is_valid, errors = validate_schema(
    df_raw,
    SCHEMA_RAW,
    stage_name="CARGA_RAW",
    logger=logger
)

if errors:
    print(f"\n⚠️  Encontrados {len(errors)} problemas:")
    for error in errors:
        print(f"  • {error}")
else:
    print("\n✅ Esquema validado correctamente")

2026-06-29 19:59:38 - ETL_Pipeline - INFO - [CARGA_RAW] Iniciando validación de esquema...
2026-06-29 19:59:38 - ETL_Pipeline - WARNING - [CARGA_RAW] ⚠️  Columnas inesperadas (se ignorarán): {'email_click_rate', 'is_premium_user', 'refund_requested', 'nps_score', 'signup_date', 'customer_id', 'churn', 'discount_used', 'coupon_code', 'email_open_rate', 'device_type', 'last_3_month_purchase_freq', 'marketing_spend_per_user', 'last_purchase_date', 'city'}


TypeError: '<' not supported between instances of 'str' and 'int'

## Etapa 3: Evaluar calidad de datos crudos

In [ ]:
metrics_raw = validate_data_quality(
    df_raw,
    stage_name="CARGA_RAW",
    logger=logger
)

print("\n📊 Resumen de Calidad (Crudo):")
for key, value in metrics_raw.items():
    if isinstance(value, float):
        print(f"  • {key}: {value:.2f}")
    else:
        print(f"  • {key}: {value}")

## Etapa 4: Cargar dataset limpio (generado por notebook 2)

In [ ]:
try:
    ruta_clean = RUTAS['data_processed'] / 'Sales_Marketing_Clean_(Codificado).csv'
    logger.info(f"Cargando dataset limpio desde: {ruta_clean}")
    
    df_clean = pd.read_csv(ruta_clean)
    logger.info(f"✅ Dataset limpio cargado: {df_clean.shape[0]} filas, {df_clean.shape[1]} columnas")
    
except FileNotFoundError:
    logger.warning("⚠️  Dataset limpio no encontrado (ejecuta notebook 2 primero)")
    df_clean = None
except Exception as e:
    logger.error(f"❌ Error al cargar dataset limpio: {str(e)}")
    df_clean = None

if df_clean is not None:
    print(df_clean.head())

## Etapa 5: Validar esquema del dataset limpio

In [ ]:
if df_clean is not None:
    is_valid_clean, errors_clean = validate_schema(
        df_clean,
        SCHEMA_CLEAN,
        stage_name="DATASET_LIMPIO",
        logger=logger
    )
    
    if errors_clean:
        print(f"\n⚠️  Encontrados {len(errors_clean)} problemas:")
        for error in errors_clean:
            print(f"  • {error}")
    else:
        print("\n✅ Dataset limpio validado correctamente")
else:
    print("⏭️  Saltando validación de dataset limpio")

## Etapa 6: Evaluar calidad de datos limpios

In [ ]:
if df_clean is not None:
    metrics_clean = validate_data_quality(
        df_clean,
        stage_name="DATASET_LIMPIO",
        logger=logger
    )
    
    print("\n📊 Resumen de Calidad (Limpio):")
    for key, value in metrics_clean.items():
        if isinstance(value, float):
            print(f"  • {key}: {value:.2f}")
        else:
            print(f"  • {key}: {value}")
    
    # Comparar antes y después
    print("\n📈 Comparación Crudo vs Limpio:")
    print(f"  • Nulls reducidos: {metrics_raw['null_ratio']:.2f}% → {metrics_clean['null_ratio']:.2f}%")
    print(f"  • Duplicados: {metrics_raw['duplicate_rows']} → {metrics_clean['duplicate_rows']}")
else:
    print("⏭️  Saltando comparación de calidad")

## Resumen Final

In [ ]:
logger.info("="*70)
logger.info("Proceso de validación completado")
logger.info("="*70)

print("\n✅ Validación completada. Revisa el archivo 'etl_pipeline.log' para más detalles.")